**Runtime Parameters**

In [0]:
from datetime import datetime, timezone

def log(msg):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC] {msg}")

log("STEP 1: Read runtime parameters - START")

dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

log(f"  table_name        : '{table_name}'")
log(f"  landing_timestamp : '{landing_timestamp}'")
log(f"  load_type         : '{load_type}'")
log(f"  merge_keys        : '{merge_keys}'")
log("STEP 1: Read runtime parameters - COMPLETE")

**Read and filter Bronze data**

In [0]:
log("STEP 2: Read and filter Bronze data - START")
log(f"  Source table : dbw_caresync_dev.bronze.{table_name}")
log(f"  Filter       : landing_timestamp = '{landing_timestamp}'")

bronze_df = spark.read.table(
    f"dbw_caresync_dev.bronze.{table_name}"
).filter(f"landing_timestamp = '{landing_timestamp}'")

# Lazy read: no Spark action here. Source record count is reported in the final cell.
log("STEP 2: Read and filter Bronze data - COMPLETE (lazy; count reported at end)")

**Add timestamp columns**

In [0]:
from pyspark.sql import functions as F

log("STEP 3: Add insert/update timestamp columns - START")

updated_df=bronze_df.withColumn("insert_timestamp", F.current_timestamp()) \
         .withColumn("update_timestamp", F.current_timestamp())

log(f"  Silver columns ({len(updated_df.columns)}): {updated_df.columns}")
log("STEP 3: Add insert/update timestamp columns - COMPLETE")

**Write the transformed DataFrame to Silver**

In [0]:
from delta.tables import DeltaTable

log("STEP 4: Write to Silver - START")

target_table = f"dbw_caresync_dev.silver.{table_name}"
load_type_normalized = (load_type or "").strip().upper()
valid_load_types = {"FULL", "APPEND", "MERGE"}

log(f"  Target table         : {target_table}")
log(f"  Normalized load_type : {load_type_normalized}")

if load_type_normalized not in valid_load_types:
    log(f"  VALIDATION FAILED: invalid load_type '{load_type}'")
    raise ValueError(
        f"Invalid load_type '{load_type}'. Expected one of: {sorted(valid_load_types)}"
    )
log("  VALIDATION PASSED: load_type is valid")

if load_type_normalized == "FULL":
    log(f"  Executing FULL load: overwrite -> {target_table}")
    updated_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    log("  FULL load write - COMPLETE")

elif load_type_normalized == "APPEND":
    log(f"  Executing APPEND load: append -> {target_table}")
    updated_df.write.format("delta").mode("append").saveAsTable(target_table)
    log("  APPEND load write - COMPLETE")

elif load_type_normalized == "MERGE":
    log("  Executing MERGE load")
    log("  MERGE validations - START")

    # Parse and validate comma-separated merge keys
    key_cols = [k.strip() for k in (merge_keys or "").split(",") if k.strip()]
    log(f"    Parsed merge key columns: {key_cols}")
    if not key_cols:
        log("    VALIDATION FAILED: merge_keys is empty")
        raise ValueError("merge_keys is required when load_type is MERGE")

    missing_keys = [k for k in key_cols if k not in updated_df.columns]
    log(f"    Merge keys missing from source: {missing_keys if missing_keys else 'none'}")
    if missing_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_keys}")

    # Fail on NULL values in any merge key column
    null_condition = " OR ".join([f"`{k}` IS NULL" for k in key_cols])
    log(f"    NULL key check condition: {null_condition}")
    null_key_count = updated_df.filter(null_condition).count()
    log(f"    Records with NULL merge keys: {null_key_count}")
    if null_key_count > 0:
        raise ValueError(
            f"Source contains {null_key_count} record(s) with NULL values in merge key "
            f"column(s) {key_cols}"
        )

    # Fail on duplicate source keys to avoid ambiguous updates
    duplicate_keys_df = updated_df.groupBy(*key_cols).count().filter("count > 1")
    duplicate_key_count = duplicate_keys_df.count()
    log(f"    Duplicate merge key values: {duplicate_key_count}")
    if duplicate_key_count > 0:
        sample = [row.asDict() for row in duplicate_keys_df.limit(5).collect()]
        log(f"    Duplicate key sample: {sample}")
        raise ValueError(
            f"Source contains {duplicate_key_count} duplicate merge key value(s) "
            f"for {key_cols}. Sample: {sample}"
        )

    log("  MERGE validations - COMPLETE (all passed)")

    if not spark.catalog.tableExists(target_table):
        log(f"  Target {target_table} does not exist: creating it from source (initial load)")
        # First run: create the Silver Delta table from the source
        updated_df.write.format("delta").saveAsTable(target_table)
        log("  Initial table creation - COMPLETE")
    else:
        log(f"  Target {target_table} exists: checking merge keys in destination")
        # Verify merge keys exist in the destination table
        target_columns = spark.table(target_table).columns
        missing_target_keys = [k for k in key_cols if k not in target_columns]
        log(f"    Merge keys missing from target: {missing_target_keys if missing_target_keys else 'none'}")
        if missing_target_keys:
            raise ValueError(
                f"merge_keys not found in destination table {target_table}: "
                f"{missing_target_keys}"
            )

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in key_cols])
        # Keep the original insert_timestamp on updates
        update_set = {
            f"`{c}`": f"s.`{c}`" for c in updated_df.columns if c != "insert_timestamp"
        }

        update_set_str = ", ".join(f"{k} = {v}" for k, v in update_set.items())
        log("  MERGE preparation (DeltaTable API):")
        log(f"    Target table     : {target_table} (alias t)")
        log("    Source           : updated_df (alias s)")
        log(f"    Merge condition  : {merge_condition}")
        log(f"    WHEN MATCHED     : UPDATE {len(update_set)} column(s), insert_timestamp preserved")
        log(f"      SET {update_set_str}")
        log("    WHEN NOT MATCHED : INSERT * (all source columns)")
        log("    Equivalent SQL (for reference only, not executed):\n"
            f"      MERGE INTO {target_table} t\n"
            f"      USING <updated_df> s\n"
            f"      ON {merge_condition}\n"
            f"      WHEN MATCHED THEN UPDATE SET {update_set_str}\n"
            f"      WHEN NOT MATCHED THEN INSERT *")

        log("  MERGE execute - START")
        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(updated_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute()
        )
        log("  MERGE execute - COMPLETE")

log("STEP 4: Write to Silver - COMPLETE")


In [0]:
record_count = updated_df.count()
log(f"  Records read from Bronze / prepared for Silver: {record_count}")
log(f"SUCCESS: {record_count} source record(s) processed for {target_table} (load_type={load_type_normalized})")
dbutils.notebook.exit(str(record_count))